# knob-senses — servidor TCP de dados


## Protocolo

TCP puro, texto, uma amostra por linha terminada em `\n`. **Toda linha tem
exatamente 9 números** separados por vírgula, sem cabeçalho e sem comentário:

```
21.946,1,100.4489,6.033333,0.4010397,79.87427,39.19426,1411.175,0.1623721
```

Essa linha é a saída real da célula 3, não um exemplo inventado.

Do lado do MCU isso é `atof()` direto, sem caso especial. A conexão é de mão
única (servidor → cliente): não há comando de entrada.

## Ordem dos eixos

É a ordem que o modelo aprendeu — **não reordene**:

| # | eixo | # | eixo | # | eixo |
|---|---|---|---|---|---|
| 0 | `temperatura_ambiente` | 3 | `hora_dec` | 6 | `ghit` |
| 1 | `doy` | 4 | `kc` | 7 | `ioeff` |
| 2 | `ghi_cs` | 5 | `angzen` | 8 | `kt` |


## 1. Configuração

Sem argumentos: o que você mudaria na linha de comando é o que se edita aqui.

- `HOST = "0.0.0.0"` aceita conexão de qualquer interface (o ESP32 chega pela
  rede local). Use `"127.0.0.1"` se quiser só a própria máquina.
- `PORT` precisa bater com o que o MCU vai discar.
- `HZ` é a cadência. `1.0` casa com `EI_CLASSIFIER_INTERVAL_MS`. Subir isso só
  serve para testar parser — o loop de 1 Hz da placa não acompanha.


In [ ]:
import csv, socket, sys, time
from pathlib import Path
import numpy as np

# ---------------------------------------------------------------------------
# Ordem EXATA dos eixos que src/main.cpp espera em readSample(). O modelo foi
# treinado nesta sequencia: reordenar aqui quebra a inferencia em silencio.
# ---------------------------------------------------------------------------
AXES = (
    "temperatura_ambiente",  # 0
    "doy",                   # 1
    "ghi_cs",                # 2
    "hora_dec",              # 3
    "kc",                    # 4
    "angzen",                # 5
    "ghit",                  # 6
    "ioeff",                 # 7
    "kt",                    # 8
)

HOST = "0.0.0.0"
PORT = 3333
HZ = 1.0

# Um stdout REDIRECIONADO (pipe, arquivo, `jupyter nbconvert --execute`) no
# Windows cai no codepage da locale (cp1252), e ai o travessao do log vira o
# byte 0x97 — que nao e UTF-8 valido. Quem le o pipe com utf-8 quebra no meio.
# Num console interativo ou no Jupyter o Python ja acerta; isto e para o resto.
# O OutStream do Jupyter nao tem reconfigure(), dai o AttributeError no guard.
for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding="utf-8", errors="replace")
    except (AttributeError, ValueError):
        pass


DATA_DIR = Path.cwd().parent / "data" / "validation"
print(f"dados em {DATA_DIR}", flush=True)

dados em c:\Users\gusta\Documents\PlatformIO\Projects\knob-senses\data


## 2. Inventário dos arquivos


In [35]:
def inventario():
    """Lista os CSVs em ordem de leitura, com contagem e colunas faltando."""
    linhas = []
    for path in sorted(DATA_DIR.glob("*.csv")):
        with path.open(newline="", encoding="utf-8-sig") as fh:
            rd = csv.DictReader(fh)
            faltando = [c for c in AXES if c not in (rd.fieldnames or [])]
            n = sum(1 for _ in rd)
        linhas.append((path.name, n, faltando))
    return linhas


_total = 0
for _nome, _n, _falta in inventario():
    _total += _n
    _aviso = f"   FALTANDO: {_falta}" if _falta else ""
    print(f"{_nome:22} {_n:>7} amostras{_aviso}", flush=True)

print(f"{'total':22} {_total:>7} amostras", flush=True)

ifpec1_2024.csv          46514 amostras
ifpec2_2024.csv          46514 amostras
ifpec3_2024.csv          46514 amostras
ifpee1_2024.csv          46514 amostras
ifpee2_2024.csv          46514 amostras
ifpee3_2024.csv          46514 amostras
total                   279084 amostras


In [36]:
def load_samples():
    """Uma amostra (tupla de 9 floats) por vez, percorrendo os CSVs em ordem."""
    for path in sorted(DATA_DIR.glob("*.csv")):
        with path.open(newline="", encoding="utf-8-sig") as fh:
            for row in csv.DictReader(fh):
                yield tuple(float(row[c]) for c in AXES)


sample_bytes = [np.array(amostra, dtype=np.float32).tobytes() for amostra in load_samples()]

[np.frombuffer(sample, dtype=np.float32) for sample in sample_bytes[0:3]]  # DEBUG: mostra as 3 primeiras amostras

[array([2.1945999e+01, 1.0000000e+00, 1.0044891e+02, 6.0333333e+00,
        4.0103966e-01, 7.9874275e+01, 3.9194260e+01, 1.4111747e+03,
        1.6237207e-01], dtype=float32),
 array([2.1938000e+01, 1.0000000e+00, 1.1997434e+02, 6.1166668e+00,
        3.5519260e-01, 7.8728027e+01, 4.1490761e+01, 1.4111747e+03,
        1.5448959e-01], dtype=float32),
 array([2.1924000e+01, 1.0000000e+00, 1.4015808e+02, 6.1999998e+00,
        2.4780591e-01, 7.7581047e+01, 3.3938793e+01, 1.4111747e+03,
        1.1444388e-01], dtype=float32)]

In [37]:

def reply(conn: socket.socket, index: int, samples=sample_bytes):
    conn.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
    with conn:
        conn.sendall(samples[index])

In [38]:
# ---------------------------------------------------------------------------
# A CELULA BLOQUEANTE. Ela NAO RETORNA: o kernel fica aqui servindo ate voce
# apertar o stop (ou Ctrl-C). Depois disso as celulas acima continuam
# disponiveis para reinspecionar os dados.
# ---------------------------------------------------------------------------
srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# SO_EXCLUSIVEADDRUSE no Windows, e NAO SO_REUSEADDR. Medido nesta maquina:
#   - com SO_REUSEADDR, um segundo processo consegue fazer bind na porta de
#     um socket que ja esta escutando, em silencio: viram dois servidores na
#     mesma porta e cada cliente cai em um.
#   - SO_EXCLUSIVEADDRUSE bloqueia isso (WSAEADDRINUSE) e, medido tambem,
#     ainda permite reabrir logo apos um cliente desconectar (TIME_WAIT).
# Nos outros sistemas SO_REUSEADDR e o certo: la ele NAO permite sequestro,
# e e ele que evita o TIME_WAIT.
_exclusivo = getattr(socket, "SO_EXCLUSIVEADDRUSE", None)
if _exclusivo is not None:
    srv.setsockopt(socket.SOL_SOCKET, _exclusivo, 1)
else:
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

try:
    srv.bind((HOST, PORT))            # erro de porta estoura AQUI, na celula
    srv.listen(1)
except OSError:
    srv.close()
    raise

# Timeout no listen so para o accept() VOLTAR ao bytecode a cada 0,5 s. Sem
# isso o botao de stop nao funciona: com o accept() bloqueado, o
# KeyboardInterrupt fica pendente e so seria levantado quando chegasse um
# cliente — ou seja, nunca, que e justamente o estado ocioso do servidor.
srv.settimeout(0.5)

print(f"Server started at {HOST}:{PORT}", flush=True)

try:
    index = 0
    while True:
            try:
              try:
                conn, addr = srv.accept()
              except socket.timeout:
                pass
              else:
                reply(conn, index)
            except OSError:
                pass
            else:
                index += 1                          # cliente desconectou no meio
except KeyboardInterrupt:
    print("\ninterrupted", flush=True)
finally:
    srv.close()
    print("socket closed", flush=True)

Server started at 0.0.0.0:3333



interrupted
socket closed
